# GLIMMER on OhioT1DM: The Paper's Other Dataset

Notebooks 02-05 ran the full v0-v3 progression on AZT1D. This notebook does the same
thing on OhioT1DM, the paper's other dataset, pulled through `azt1d.metabonet` instead
of a dataset-specific loader. Same code throughout, `train_subject_model`,
`search_patient_weights`, and everything else don't know or care which dataset a
DataFrame came from.

**Worth knowing going in:** the (3.29, 2.38) weights used as "the paper's published
weights" throughout this whole project were themselves found by a GA search *on
OhioT1DM* (their Table 3 caption says so directly), not AZT1D. So this notebook's own
GA search is being run on the same dataset those reference numbers came from, a more
direct test of whether we can reproduce the paper's actual search than notebook 05's
AZT1D run was.

**What this covers:** v0 (plain baseline), v1 (fixed weighted loss), v2 (clinical
metrics), and v3 (the real per-patient GA search) for both CNN-LSTM and Transformer,
plus a direct comparison against the AZT1D results from notebooks 02-05.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from azt1d import plotting
from azt1d import reference as ref
from azt1d.metabonet import load_source
from azt1d.glimmer.train import run_with_checkpoints, train_prepared_model, prepare_subject_data, region_errors
from azt1d.glimmer.ga import search_patient_weights
from azt1d.glimmer.clinical import dysglycemia_event_metrics, clarke_zone_percentages
from azt1d.glimmer import checkpoint

plotting.apply_style()
pd.set_option("display.max_columns", None)


## 1. Load OhioT1DM

Pulled entirely through `azt1d.metabonet.load_source`, validated earlier against
AZT1D's own raw files (exact match on CGM, bolus, and carbs; basal matches after a unit
conversion MetaboNet needs). No OhioT1DM-specific loading code exists anywhere in this
project, this is the same function any of MetaboNet's 14 datasets goes through.


In [ ]:
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "processed" / "checkpoints"
GLIMMER_WEIGHTS = ref.GLIMMER_PAPER_WEIGHTS["cnn_lstm"]

df = load_source("OhioT1DM")
print(f"Loaded {len(df):,} rows across {df['subject_id'].nunique()} subjects: {sorted(df['subject_id'].unique())}")


## 2. CNN-LSTM: baseline and weighted loss


In [ ]:
summary_v0, results_v0 = run_with_checkpoints(
    df, CHECKPOINT_DIR / "ohiot1dm_cnn_lstm_v0", epochs=30, region_weights=None)
summary_v1, results_v1 = run_with_checkpoints(
    df, CHECKPOINT_DIR / "ohiot1dm_cnn_lstm_v1", epochs=30, region_weights=GLIMMER_WEIGHTS)

print(f"v0 (plain loss):         RMSE {summary_v0['rmse'].mean():.2f} +/- {summary_v0['rmse'].std():.2f} mg/dL   "
      f"MAE {summary_v0['mae'].mean():.2f} +/- {summary_v0['mae'].std():.2f} mg/dL")
print(f"v1 (fixed avg. weights): RMSE {summary_v1['rmse'].mean():.2f} +/- {summary_v1['rmse'].std():.2f} mg/dL   "
      f"MAE {summary_v1['mae'].mean():.2f} +/- {summary_v1['mae'].std():.2f} mg/dL")
print()
print("Paper's CNN-LSTM baseline on OhioT1DM:  RMSE 31.98 +/- 4.15 mg/dL, MAE 23.00 +/- 2.87 mg/dL")
print("Paper's GLIMMER(CNN-LSTM) on OhioT1DM:  RMSE 23.97 +/- 3.77 mg/dL, MAE 15.83 +/- 2.09 mg/dL")


## 3. Does the clinical picture match what we found on AZT1D?

Notebook 04 found v1 trades precision for recall and cuts dangerous missed-detection
errors on AZT1D, even though its plain RMSE is worse. Same check here.


In [ ]:
def cohort_event_metrics(results):
    all_true = np.concatenate([r.y_test for r in results.values()])
    all_pred = np.concatenate([r.y_pred for r in results.values()])
    return dysglycemia_event_metrics(all_true, all_pred)

def cohort_zone_percentages(results):
    all_true = np.concatenate([r.y_test for r in results.values()])
    all_pred = np.concatenate([r.y_pred for r in results.values()])
    return clarke_zone_percentages(all_true, all_pred)

event_compare = pd.DataFrame({"v0": cohort_event_metrics(results_v0), "v1": cohort_event_metrics(results_v1)}).loc[["precision", "recall", "f1"]]
zone_compare = pd.DataFrame({"v0": cohort_zone_percentages(results_v0), "v1": cohort_zone_percentages(results_v1)}).reindex(["A", "B", "C", "D", "E"])

print("Event detection:")
display(event_compare.round(3))
print("Clarke Error Grid zones (%):")
display(zone_compare.round(2))


## 4. CNN-LSTM: the real per-patient GA search

Same scoped-down search as notebook 05 (population 6, 6 generations, capped epochs per
candidate), run on OhioT1DM directly, the dataset the paper's own (3.29, 2.38) weights
actually came from.


In [ ]:
GA_CHECKPOINT_DIR = CHECKPOINT_DIR / "ohiot1dm_ga_cnn_lstm"

ga_results = {}
prepared_data = {}

for sid in sorted(df["subject_id"].unique()):
    df_subject = df[df["subject_id"] == sid].reset_index(drop=True)
    data = prepare_subject_data(df_subject)
    prepared_data[sid] = data

    cached = checkpoint.load_ga_result(GA_CHECKPOINT_DIR, sid)
    if cached is not None:
        ga_results[sid] = cached
        print(f"Subject {sid}: loaded from checkpoint (w_hypo={cached['best_weights']['w_hypo']:.2f}, "
              f"w_hyper={cached['best_weights']['w_hyper']:.2f}), val RMSE {cached['best_fitness']:.2f} mg/dL")
        continue

    result = search_patient_weights(data, architecture="cnn_lstm", seed=0)
    checkpoint.save_ga_result(GA_CHECKPOINT_DIR, sid, result.best_weights, result.best_fitness,
                               result.history, result.n_evaluations)
    ga_results[sid] = {
        "best_weights": result.best_weights, "best_fitness": result.best_fitness,
        "history": result.history, "n_evaluations": result.n_evaluations,
    }
    print(f"Subject {sid}: found (w_hypo={result.best_weights['w_hypo']:.2f}, "
          f"w_hyper={result.best_weights['w_hyper']:.2f}), val RMSE {result.best_fitness:.2f} mg/dL "
          f"({result.n_evaluations} candidates tried)")


In [ ]:
weights_table = pd.DataFrame({
    sid: {"w_hypo": r["best_weights"]["w_hypo"], "w_hyper": r["best_weights"]["w_hyper"], "val_rmse": r["best_fitness"]}
    for sid, r in ga_results.items()
}).T
weights_table.index.name = "subject_id"
print("Per-patient weights found on OhioT1DM, vs. the paper's own average from this same dataset (3.29, 2.38):")
weights_table.round(2)


## 5. Final CNN-LSTM models with per-patient weights


In [ ]:
V3_CHECKPOINT_DIR = CHECKPOINT_DIR / "ohiot1dm_cnn_lstm_v3_tuned"

results_v3 = {}
rows = []
for sid, data in prepared_data.items():
    cached = checkpoint.load_result(V3_CHECKPOINT_DIR, sid)
    if cached is not None:
        result = cached
        print(f"Subject {sid}: loaded from checkpoint (RMSE={result.rmse:.2f} mg/dL)")
    else:
        weights = ga_results[sid]["best_weights"]
        result = train_prepared_model(data, architecture="cnn_lstm", epochs=30, region_weights=weights)
        checkpoint.save_result(V3_CHECKPOINT_DIR, result)
        print(f"Subject {sid}: trained and checkpointed (RMSE={result.rmse:.2f} mg/dL)")

    results_v3[sid] = result
    rows.append({"subject_id": sid, "rmse": result.rmse, "mae": result.mae, "n_test": result.n_test})

summary_v3 = pd.DataFrame(rows).set_index("subject_id")
print(f"v3 (per-patient weights): RMSE {summary_v3['rmse'].mean():.2f} +/- {summary_v3['rmse'].std():.2f} mg/dL   "
      f"MAE {summary_v3['mae'].mean():.2f} +/- {summary_v3['mae'].std():.2f} mg/dL")


## 6. Transformer: baseline, weighted loss, GA search, tuned models

Same four steps, second architecture. Checkpointed the same way as everything else.


In [ ]:
summary_tf_v0, results_tf_v0 = run_with_checkpoints(
    df, CHECKPOINT_DIR / "ohiot1dm_cnn_transformer_v0", epochs=30, architecture="cnn_transformer", region_weights=None)
summary_tf_v1, results_tf_v1 = run_with_checkpoints(
    df, CHECKPOINT_DIR / "ohiot1dm_cnn_transformer_v1", epochs=30, architecture="cnn_transformer", region_weights=GLIMMER_WEIGHTS)

print(f"Transformer v0: RMSE {summary_tf_v0['rmse'].mean():.2f} +/- {summary_tf_v0['rmse'].std():.2f} mg/dL   "
      f"MAE {summary_tf_v0['mae'].mean():.2f} +/- {summary_tf_v0['mae'].std():.2f} mg/dL")
print(f"Transformer v1: RMSE {summary_tf_v1['rmse'].mean():.2f} +/- {summary_tf_v1['rmse'].std():.2f} mg/dL   "
      f"MAE {summary_tf_v1['mae'].mean():.2f} +/- {summary_tf_v1['mae'].std():.2f} mg/dL")


In [ ]:
GA_TRANSFORMER_CHECKPOINT_DIR = CHECKPOINT_DIR / "ohiot1dm_ga_cnn_transformer"

ga_results_tf = {}
for sid in sorted(df["subject_id"].unique()):
    data = prepared_data[sid]

    cached = checkpoint.load_ga_result(GA_TRANSFORMER_CHECKPOINT_DIR, sid)
    if cached is not None:
        ga_results_tf[sid] = cached
        print(f"Subject {sid}: loaded from checkpoint (w_hypo={cached['best_weights']['w_hypo']:.2f}, "
              f"w_hyper={cached['best_weights']['w_hyper']:.2f}), val RMSE {cached['best_fitness']:.2f} mg/dL")
        continue

    result = search_patient_weights(data, architecture="cnn_transformer", seed=0)
    checkpoint.save_ga_result(GA_TRANSFORMER_CHECKPOINT_DIR, sid, result.best_weights, result.best_fitness,
                               result.history, result.n_evaluations)
    ga_results_tf[sid] = {
        "best_weights": result.best_weights, "best_fitness": result.best_fitness,
        "history": result.history, "n_evaluations": result.n_evaluations,
    }
    print(f"Subject {sid}: found (w_hypo={result.best_weights['w_hypo']:.2f}, "
          f"w_hyper={result.best_weights['w_hyper']:.2f}), val RMSE {result.best_fitness:.2f} mg/dL "
          f"({result.n_evaluations} candidates tried)")


In [ ]:
V3_TRANSFORMER_CHECKPOINT_DIR = CHECKPOINT_DIR / "ohiot1dm_cnn_transformer_v3_tuned"

results_v3_tf = {}
rows = []
for sid, data in prepared_data.items():
    cached = checkpoint.load_result(V3_TRANSFORMER_CHECKPOINT_DIR, sid)
    if cached is not None:
        result = cached
        print(f"Subject {sid}: loaded from checkpoint (RMSE={result.rmse:.2f} mg/dL)")
    else:
        weights = ga_results_tf[sid]["best_weights"]
        result = train_prepared_model(data, architecture="cnn_transformer", epochs=30, region_weights=weights)
        checkpoint.save_result(V3_TRANSFORMER_CHECKPOINT_DIR, result)
        print(f"Subject {sid}: trained and checkpointed (RMSE={result.rmse:.2f} mg/dL)")

    results_v3_tf[sid] = result
    rows.append({"subject_id": sid, "rmse": result.rmse, "mae": result.mae, "n_test": result.n_test})

summary_v3_tf = pd.DataFrame(rows).set_index("subject_id")
print(f"Transformer v3: RMSE {summary_v3_tf['rmse'].mean():.2f} +/- {summary_v3_tf['rmse'].std():.2f} mg/dL   "
      f"MAE {summary_v3_tf['mae'].mean():.2f} +/- {summary_v3_tf['mae'].std():.2f} mg/dL")


## 7. The complete OhioT1DM picture


In [ ]:
full_comparison = pd.DataFrame({
    "CNN-LSTM v0": [summary_v0["rmse"].mean(), summary_v0["mae"].mean()],
    "CNN-LSTM v1": [summary_v1["rmse"].mean(), summary_v1["mae"].mean()],
    "CNN-LSTM v3": [summary_v3["rmse"].mean(), summary_v3["mae"].mean()],
    "Transformer v0": [summary_tf_v0["rmse"].mean(), summary_tf_v0["mae"].mean()],
    "Transformer v1": [summary_tf_v1["rmse"].mean(), summary_tf_v1["mae"].mean()],
    "Transformer v3": [summary_v3_tf["rmse"].mean(), summary_v3_tf["mae"].mean()],
}, index=["RMSE", "MAE"]).round(2)
full_comparison


In [ ]:
def region_summary(results):
    rows = []
    for sid, r in results.items():
        re = region_errors(r.y_test, r.y_pred)
        for region in ("hypo", "normal", "hyper"):
            rows.append({"subject_id": sid, "region": region, "mae": re[region]["mae"]})
    return pd.DataFrame(rows).groupby("region")["mae"].mean().reindex(["hypo", "normal", "hyper"])

region_compare = pd.DataFrame({
    "v0": region_summary(results_v0), "v1": region_summary(results_v1), "v3": region_summary(results_v3),
})
region_compare.round(2)


## 8. Significance, both architectures


In [ ]:
paired = summary_v0[["rmse"]].rename(columns={"rmse": "v0"}).join(
    summary_v1[["rmse"]].rename(columns={"rmse": "v1"})).join(
    summary_v3[["rmse"]].rename(columns={"rmse": "v3"}))
paired_tf = summary_tf_v0[["rmse"]].rename(columns={"rmse": "v0"}).join(
    summary_tf_v1[["rmse"]].rename(columns={"rmse": "v1"})).join(
    summary_v3_tf[["rmse"]].rename(columns={"rmse": "v3"}))

for label, table, a, b in [
    ("CNN-LSTM v0 vs v1", paired, "v0", "v1"),
    ("CNN-LSTM v0 vs v3", paired, "v0", "v3"),
    ("Transformer v0 vs v1", paired_tf, "v0", "v1"),
    ("Transformer v0 vs v3", paired_tf, "v0", "v3"),
]:
    stat, p = wilcoxon(table[a], table[b])
    direction = "lower" if table[b].mean() < table[a].mean() else "higher"
    print(f"{label}: {b} RMSE is {direction} on average, p-value = {p:.4g} "
          f"({'significant' if p < 0.05 else 'not significant'} at alpha=0.05)")


## 9. OhioT1DM vs. AZT1D, side by side

The actual point of building the MetaboNet path: the same method, the same code, on two
different real-world populations.


In [ ]:
azt1d_cnn_lstm = {
    "v0": {"rmse": 31.54, "mae": 24.01}, "v1": {"rmse": 41.18, "mae": 32.73}, "v3": {"rmse": 37.44, "mae": 28.97},
}
ohio_cnn_lstm = {
    "v0": {"rmse": summary_v0["rmse"].mean(), "mae": summary_v0["mae"].mean()},
    "v1": {"rmse": summary_v1["rmse"].mean(), "mae": summary_v1["mae"].mean()},
    "v3": {"rmse": summary_v3["rmse"].mean(), "mae": summary_v3["mae"].mean()},
}

cross_dataset = pd.DataFrame({
    "AZT1D RMSE": [azt1d_cnn_lstm[v]["rmse"] for v in ("v0", "v1", "v3")],
    "OhioT1DM RMSE": [ohio_cnn_lstm[v]["rmse"] for v in ("v0", "v1", "v3")],
    "AZT1D MAE": [azt1d_cnn_lstm[v]["mae"] for v in ("v0", "v1", "v3")],
    "OhioT1DM MAE": [ohio_cnn_lstm[v]["mae"] for v in ("v0", "v1", "v3")],
}, index=["v0", "v1", "v3"]).round(2)
cross_dataset


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
versions = ["v0", "v1", "v3"]
x = np.arange(len(versions))
width = 0.35
ax.bar(x - width/2, cross_dataset["AZT1D RMSE"], width, color=plotting.CATEGORICAL[0], label="AZT1D")
ax.bar(x + width/2, cross_dataset["OhioT1DM RMSE"], width, color=plotting.CATEGORICAL[1], label="OhioT1DM")
ax.set_xticks(x)
ax.set_xticklabels(versions)
ax.set_ylabel("RMSE (mg/dL), CNN-LSTM")
ax.set_title("Same method, two datasets: does the pattern hold?")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 10. Where this leaves the full replication

Once every cell above has actually run, this section should state directly: does
OhioT1DM (the dataset the paper's own reference weights came from) show the same
"per-patient tuning helps but doesn't beat baseline" pattern AZT1D did, or does it
actually reproduce the paper's claimed improvement here. That's the real test this
notebook was built for. Read Sections 7-9 directly rather than assuming the answer
here, this is a placeholder until the run is done.
